In [7]:
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [8]:
def get_train_data():
    return np.load('../../../data/MNIST/train_data.npy'), np.load('../../../data/MNIST/train_labels.npy')

def get_test_data():
    return np.load('../../../data/MNIST/test_data.npy'), np.load('../../../data/MNIST/test_labels.npy')

In [9]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        # network parameters
        hidden_units = 256
        dropout = 0.45
        input_size = 784
        num_labels = 10
        # Define the layers
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_units, hidden_units)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_units, num_labels)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [10]:
def train_model(model, X_train, y_train, loss_fn, optimizer, epochs, batch_size):
    # Convert numpy arrays to PyTorch tensors
    X_train = torch.from_numpy(X_train).float()
    y_train = torch.from_numpy(y_train).long()
    
    # Create a TensorDataset
    train_dataset = TensorDataset(X_train, y_train)

    # Create a DataLoader for the training dataset with the defined batch size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    
    for epoch in range(epochs):
        total_correct = 0
        total_samples = 0
        total_loss = 0
        
        for i, (data, labels) in enumerate(train_loader):
            # Forward pass
            outputs = model(data)
            loss = loss_fn(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Compute training accuracy
            _, predicted = torch.max(outputs.data, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            total_loss += loss.item()

        # Print loss and accuracy at the end of the epoch
        epoch_loss = total_loss / (i + 1)
        epoch_accuracy = total_correct / total_samples
        print('Epoch [{}/{}], Loss: {:.4f}, Accuracy: {:.4f}'
            .format(epoch+1, epochs, epoch_loss, epoch_accuracy))
        
    return model

In [11]:
X_train, y_train = get_train_data()
y_train = np.argmax(y_train, axis=1)

In [12]:
model = Model()
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 1,
      }
  },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 2,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "pytorch",
      "params": {
        "loss": nn.CrossEntropyLoss(),
        "optimizer": optim.Adam(model.parameters(), lr=0.001),
      }
    },
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
  }
}

In [13]:
model = train_model(model, X_train, y_train, config["DL"]["lib"]["params"]["loss"], config["DL"]["lib"]["params"]["optimizer"], config["DL"]["epochs"], config["DL"]["batch_size"])

Epoch [1/20], Loss: 0.4504, Accuracy: 0.8661
Epoch [2/20], Loss: 0.2003, Accuracy: 0.9401
Epoch [3/20], Loss: 0.1543, Accuracy: 0.9539
Epoch [4/20], Loss: 0.1291, Accuracy: 0.9607
Epoch [5/20], Loss: 0.1120, Accuracy: 0.9654
Epoch [6/20], Loss: 0.1045, Accuracy: 0.9683
Epoch [7/20], Loss: 0.0915, Accuracy: 0.9708
Epoch [8/20], Loss: 0.0875, Accuracy: 0.9726
Epoch [9/20], Loss: 0.0838, Accuracy: 0.9739
Epoch [10/20], Loss: 0.0781, Accuracy: 0.9751
Epoch [11/20], Loss: 0.0750, Accuracy: 0.9763
Epoch [12/20], Loss: 0.0673, Accuracy: 0.9779
Epoch [13/20], Loss: 0.0644, Accuracy: 0.9799
Epoch [14/20], Loss: 0.0645, Accuracy: 0.9789
Epoch [15/20], Loss: 0.0647, Accuracy: 0.9797
Epoch [16/20], Loss: 0.0591, Accuracy: 0.9808
Epoch [17/20], Loss: 0.0588, Accuracy: 0.9805
Epoch [18/20], Loss: 0.0558, Accuracy: 0.9823
Epoch [19/20], Loss: 0.0564, Accuracy: 0.9818
Epoch [20/20], Loss: 0.0550, Accuracy: 0.9820


In [14]:
def evaluate_model(model, X_test, y_test, batch_size):
    # Convert numpy arrays to PyTorch tensors
    X_test = torch.from_numpy(X_test).float()
    y_test = torch.from_numpy(y_test).long()
    
    # Create a TensorDataset
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    total_correct = 0
    total_samples = 0
    
    for i, (data, labels) in enumerate(test_loader):
        # Forward pass
        outputs = model(data)

        # Compute training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples

In [15]:
X_test, y_test = get_test_data()
y_test = np.argmax(y_test, axis=1)

In [16]:
acc = evaluate_model(model, X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))


Test accuracy: 96.9%
